# Modeling

This notebook starts with a deliberately simple baseline model. The goal of the first baseline is not to get the best score immediately. It is to create a trustworthy reference point using stable, high-coverage features so every later model improvement can be judged against something clean.

## Why 2022 disappeared

The processed master dataset still contains 2022 rows. The rows disappear during modeling when `dropna()` is applied to a feature set that includes `WindSpeed`.

`WindSpeed` is missing for all 2022 rows, so this kind of filter removes the whole year:

```python
model_df = master_df.dropna(subset=features + ["SolarGeneration"])
```

For the first baseline, exclude `WindSpeed`. After the baseline is working, test `WindSpeed` as a controlled second experiment using an imputed value plus a `WindSpeed_missing` flag.

In [24]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
MASTER_PATH = PROJECT_ROOT / "data" / "processed" / "master_dataset.parquet"

master_df = pd.read_parquet(MASTER_PATH)
master_df["Timestamp"] = pd.to_datetime(master_df["Timestamp"])

master_df.shape

(2731946, 17)

In [25]:
coverage_cols = [
    "SolarGeneration",
    "Ghi",
    "CloudOpacity",
    "AirTemperature",
    "RelativeHumidity",
    "WindSpeed",
    "kWp",
]

yearly_coverage = (
    master_df
    .groupby(master_df["Timestamp"].dt.year)[coverage_cols]
    .apply(lambda group: group.notna().mean().round(3))
)

yearly_rows = master_df.groupby(master_df["Timestamp"].dt.year).size()

display(yearly_rows.rename("rows"))
display(yearly_coverage)

Timestamp
2020     845495
2021    1430835
2022     455616
Name: rows, dtype: int64

,SolarGeneration,Ghi,CloudOpacity,AirTemperature,RelativeHumidity,WindSpeed,kWp
Timestamp,,,,,,,
2020,0.872,0.962,0.962,0.944,0.944,0.927,0.559
2021,0.887,1.000,1.000,0.752,0.752,0.536,0.599
2022,0.921,0.999,0.999,0.504,0.504,0.000,0.595


In [26]:
feature_sets_to_check = {
    "with_wind": [
        "SolarGeneration",
        "Ghi",
        "CloudOpacity",
        "AirTemperature",
        "RelativeHumidity",
        "WindSpeed",
    ],
    "no_wind": [
        "SolarGeneration",
        "Ghi",
        "CloudOpacity",
        "AirTemperature",
        "RelativeHumidity",
    ],
    "solar_irradiance_only": [
        "SolarGeneration",
        "Ghi",
        "CloudOpacity",
    ],
}

for name, cols in feature_sets_to_check.items():
    filtered = master_df.dropna(subset=cols)
    print(f"\n{name}")
    print(filtered.groupby(filtered["Timestamp"].dt.year).size())


with_wind
Timestamp
2020    681005
2021    669570
dtype: int64

no_wind
Timestamp
2020    694505
2021    948938
2022    210663
dtype: int64

solar_irradiance_only
Timestamp
2020     708085
2021    1268626
2022     419154
dtype: int64


## First baseline

A first baseline is the simplest credible model that proves the modeling setup is working. It should use features with good coverage, preserve the time-based test period, and avoid clever fixes that hide data-quality issues.

For this project, the first baseline excludes `WindSpeed` because it deletes all 2022 rows if used with strict `dropna()`. The baseline keeps 2022 as the holdout period so the model is evaluated on future data rather than random rows from the same time period.

In [27]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Hour"] = df["Timestamp"].dt.hour
    df["Month"] = df["Timestamp"].dt.month
    df["DayOfYear"] = df["Timestamp"].dt.dayofyear
    df["DayOfWeek"] = df["Timestamp"].dt.dayofweek
    return df


model_df = add_time_features(master_df)

baseline_features = [
    "Ghi",
    "CloudOpacity",
    "AirTemperature",
    "RelativeHumidity",
    "Hour",
    "Month",
    "DayOfYear",
    "DayOfWeek",
    "CampusKey",
    "SiteKey",
    "kWp",
    "has_capacity_data",
]

target = "SolarGeneration"

model_df = model_df.dropna(
    subset=[
        target,
        "Ghi",
        "CloudOpacity",
    ]
)

train_df = model_df[model_df["Timestamp"] < "2022-01-01"].copy()
test_df = model_df[model_df["Timestamp"] >= "2022-01-01"].copy()

print("Train date range:", train_df["Timestamp"].min(), "to", train_df["Timestamp"].max())
print("Test date range:", test_df["Timestamp"].min(), "to", test_df["Timestamp"].max())
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

Train date range: 2020-01-01 00:15:00 to 2021-12-31 23:45:00
Test date range: 2022-01-01 00:00:00 to 2022-04-23 22:00:00
Train rows: 1976711
Test rows: 419154


In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

numeric_features = [
    "Ghi",
    "CloudOpacity",
    "AirTemperature",
    "RelativeHumidity",
    "Hour",
    "Month",
    "DayOfYear",
    "DayOfWeek",
    "kWp",
    "has_capacity_data",
]

categorical_features = [
    "CampusKey",
    "SiteKey",
]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "cat",
            # Added sparse_output=False to prevent the dense matrix crash
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ]
)

baseline_model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "model",
            HistGradientBoostingRegressor(
                max_iter=150,
                learning_rate=0.08,
                random_state=42,
            ),
        ),
    ]
)

X_train = train_df[baseline_features]
y_train = train_df[target]
X_test = test_df[baseline_features]
y_test = test_df[target]

# This will now run smoothly without the TypeError
baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_test)

In [32]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def regression_metrics(y_true, y_pred) -> dict:
    rmse = root_mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse,
        "R2": r2_score(y_true, y_pred),
    }

baseline_metrics = regression_metrics(y_test, baseline_preds)
baseline_metrics

{'MAE': 0.8707071472559275,
 'RMSE': 2.5936967797889108,
 'R2': 0.9251436572350815}

## WindSpeed experiment

`WindSpeed_missing` can be useful, but it should not be part of the first baseline. Because `WindSpeed` is missing for all 2022 rows, the missingness flag may learn data-availability patterns instead of weather physics. Treat it as a second experiment and compare it against the baseline.

If this version improves 2022 performance, keep it as a candidate. If it only improves because it separates years or campuses, prefer the cleaner baseline.

In [34]:
wind_df = add_time_features(master_df)
wind_df["WindSpeed_missing"] = wind_df["WindSpeed"].isna().astype("int8")

wind_df = wind_df.dropna(
    subset=[
        target,
        "Ghi",
        "CloudOpacity",
    ]
)

wind_train_df = wind_df[wind_df["Timestamp"] < "2022-01-01"].copy()
wind_test_df = wind_df[wind_df["Timestamp"] >= "2022-01-01"].copy()

wind_features = baseline_features + [
    "WindSpeed",
    "WindSpeed_missing",
]

wind_numeric_features = numeric_features + [
    "WindSpeed",
    "WindSpeed_missing",
]

wind_preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            wind_numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ]
)

wind_model = Pipeline(
    steps=[
        ("preprocess", wind_preprocess),
        (
            "model",
            HistGradientBoostingRegressor(
                max_iter=150,
                learning_rate=0.08,
                random_state=42,
            ),
        ),
    ]
)

wind_model.fit(wind_train_df[wind_features], wind_train_df[target])
wind_preds = wind_model.predict(wind_test_df[wind_features])

comparison = pd.DataFrame(
    [
        regression_metrics(y_test, baseline_preds),
        regression_metrics(wind_test_df[target], wind_preds),
    ],
    index=[
        "baseline_no_wind",
        "wind_with_missing_flag",
    ],
)

comparison

,MAE,RMSE,R2
baseline_no_wind,0.870707,2.593697,0.925144
wind_with_missing_flag,0.866716,2.552200,0.927520


## What to do after the first baseline

1. Confirm the split is valid: train on 2020-2021 and test on 2022.
2. Save the baseline metrics. Every later model must beat these numbers on the same 2022 test set.
3. Inspect errors by campus, site, hour, and month. This tells you where the model is weak.
4. Add features one group at a time: capacity features, weather features, lag features, rolling irradiance features, then missingness flags.
5. Compare each experiment against the baseline. Keep a feature only if it improves holdout performance and makes physical/business sense.
6. Once the feature set is stable, tune the model and document the final model choice.

In [35]:
results = test_df[[
    "CampusKey",
    "SiteKey",
    "Timestamp",
    target,
]].copy()

results["Prediction"] = baseline_preds
results["Error"] = results[target] - results["Prediction"]
results["AbsoluteError"] = results["Error"].abs()

campus_error = (
    results
    .groupby("CampusKey")
    .agg(
        rows=(target, "size"),
        mae=("AbsoluteError", "mean"),
        bias=("Error", "mean"),
    )
    .sort_values("mae", ascending=False)
)

hour_error = (
    results
    .groupby(results["Timestamp"].dt.hour)
    .agg(
        rows=(target, "size"),
        mae=("AbsoluteError", "mean"),
        bias=("Error", "mean"),
    )
    .sort_index()
)

display(campus_error)
display(hour_error)

,rows,mae,bias
CampusKey,,,
4,10043,1.599093,0.581900
3,76479,1.085694,0.116709
1,273609,0.876620,-0.325969
2,48887,0.468493,-0.002771
5,10136,0.307165,-0.173253


,rows,mae,bias
Timestamp,,,
0,18984,0.035567,0.026088
1,18984,0.035282,0.028034
2,18984,0.034627,0.029405
3,18984,0.034317,0.030342
4,18984,0.034418,0.030436
5,18984,0.034918,0.030986
6,18984,0.111105,-0.050615
7,9188,0.545639,-0.392176
8,17259,0.866810,-0.243166
